(cap-osmnx)=
# Aquisição e Modelagem de Redes com OpenStreetMap e OSMnx

:::{admonition} Objetivos de Aprendizagem
:class: important

Ao final deste capítulo, você será capaz de:

1. **Descrever** o modelo de dados do OpenStreetMap (nós, *ways*, relações e *tags*) e o modo como ele é consultado pela API Overpass.
2. **Explicar** como a OSMnx converte *ways* em um multidígrafo $G = (V, E)$, incluindo a regra de simplificação topológica e o tratamento de vias de mão única.
3. **Delimitar** áreas de estudo por geocodificação, ponto e raio, e **extrair** feições geográficas (edificações, parques, pontos de ônibus, equipamentos) com filtros de *tags*.
4. **Obter** redes viárias para diferentes modos de deslocamento e **comparar** suas propriedades.
5. **Projetar** grafos em sistemas de coordenadas métricas e **converter** grafos entre as representações NetworkX e GeoPandas.
6. **Diagnosticar e tratar** atributos heterogêneos, como listas e textos, em nós e arestas antes de análises quantitativas.
:::

O {ref}`cap-fundamentos-grafos` construiu grafos manualmente, nó a nó. Redes reais, com milhares de interseções, exigem uma fonte de dados abrangente e um procedimento sistemático de modelagem. Neste capítulo, usamos o **OpenStreetMap** (OSM) como fonte e a biblioteca **OSMnx** {cite:p}`boeing2017,boeing2025` como ferramenta de aquisição e modelagem.

A OSMnx é construída sobre a NetworkX, para a estrutura de grafo, e sobre a GeoPandas, para a geometria. Com poucas linhas de código, ela permite baixar redes viárias e feições urbanas de praticamente qualquer lugar do mundo, convertê-las em grafos analisáveis e exportá-las para formatos SIG. O caminho percorrido pelos dados, do mapeamento colaborativo ao grafo pronto para análise, é o tema deste capítulo.

## O projeto OpenStreetMap

O OpenStreetMap é um projeto colaborativo, iniciado em 2004, que produz uma base de dados geográficos aberta e editável por qualquer pessoa {cite:p}`haklay2008`. Voluntários mapeiam vias, edificações, limites administrativos, pontos de interesse e muitas outras feições a partir de levantamentos de campo, trilhas de GPS e imagens aéreas ou de satélite. Os dados são distribuídos sob a licença *Open Database License* (ODbL), que exige atribuição e o compartilhamento de obras derivadas sob a mesma licença.

### Modelo de dados

O OSM representa o mundo com apenas três primitivas:

- **nós** (*nodes*): pontos com identificador, latitude e longitude;
- **caminhos** (*ways*): sequências ordenadas de nós, que representam linhas (uma rua, um rio) ou, quando fechadas, polígonos (uma edificação, um parque);
- **relações** (*relations*): agrupamentos de nós, caminhos e outras relações, usados para feições complexas, como linhas de ônibus, restrições de conversão e limites administrativos.

A semântica de cada elemento é dada por **tags**, pares `chave=valor` que o descrevem. A {numref}`tab-tags-osm` apresenta algumas chaves recorrentes neste livro.

:::{table} Chaves (*keys*) do OpenStreetMap usadas com frequência na análise de redes urbanas.
:name: tab-tags-osm

| Chave | Exemplos de valores | Significado |
|---|---|---|
| `highway` | `primary`, `residential`, `footway`, `bus_stop` | Vias e elementos associados à circulação |
| `building` | `yes`, `residential`, `commercial` | Edificações |
| `amenity` | `school`, `hospital`, `parking` | Serviços e equipamentos |
| `leisure` | `park`, `playground` | Áreas de lazer |
| `landuse`, `natural` | `forest`, `water` | Uso do solo e elementos naturais |
| `waterway` | `river`, `stream` | Cursos d'água |
| `oneway`, `maxspeed`, `lanes` | `yes`, `50`, `2` | Atributos de circulação de uma via |

:::

A documentação completa das *tags* está na [Wiki do OpenStreetMap](https://wiki.openstreetmap.org/wiki/Map_features), em especial nas páginas das chaves [`highway`](https://wiki.openstreetmap.org/wiki/Key:highway), [`building`](https://wiki.openstreetmap.org/wiki/Key:building), [`amenity`](https://wiki.openstreetmap.org/wiki/Key:amenity) e [`landuse`](https://wiki.openstreetmap.org/wiki/Key:landuse).

:::{warning}
O OSM é um produto colaborativo e **heterogêneo**: a completude e a atualização dos dados variam entre cidades, e até entre bairros de uma mesma cidade. Atributos como `maxspeed` e `lanes` são opcionais e frequentemente ausentes. Toda análise deve incluir uma avaliação da qualidade e da completude dos dados na área de estudo, e os resultados devem registrar a data da extração.
:::

### Do OSM ao grafo: a modelagem da OSMnx

A OSMnx consulta a **API Overpass**, um serviço de leitura do OSM, para obter todos os *ways* com a chave `highway` que satisfazem um filtro associado ao tipo de rede desejado. Em seguida, transforma esses *ways* em um **multidígrafo** $G = (V, E)$ seguindo as regras abaixo {cite:p}`boeing2017`.

1. **Arestas orientadas.** Cada par de nós OSM consecutivos em um *way* gera um arco. Se a via é de mão dupla, são criados os arcos $(u, v)$ e $(v, u)$; se é de mão única (`oneway=yes`), apenas o arco no sentido permitido.
2. **Multiplicidade.** Como duas vias distintas podem ligar as mesmas interseções, cada aresta é identificada pela tripla $(u, v, k)$, como nos multigrafos do {ref}`cap-fundamentos-grafos`.
3. **Simplificação topológica.** Os *ways* do OSM contêm muitos nós que apenas descrevem a *forma* da via (curvas), sem representar interseções. A OSMnx remove esses nós intersticiais e funde as arestas adjacentes, preservando a geometria completa no atributo `geometry`. Seja $N(v)$ o conjunto de vizinhos distintos de $v$, sejam eles predecessores ou sucessores. Um nó é mantido, e chamado de **ponto final**, se

$$
|N(v)| \neq 2 \;\;\lor\;\; \deg(v) \notin \{2, 4\} \;\;\lor\;\; \deg^{-}(v) = 0 \;\;\lor\;\; \deg^{+}(v) = 0 \;\;\lor\;\; v \in N(v).
$$ (eq-regra-simplificacao)

   Em palavras: são mantidos os nós que ligam um número de vizinhos diferente de dois (interseções e extremidades), os que não ligam exatamente uma via de mão única ($\deg = 2$) ou de mão dupla ($\deg = 4$), as extremidades de vias sem saída e os nós com autolaço. Os demais nós, que apenas descrevem a forma de uma via contínua, são removidos.

4. **Comprimento.** O atributo `length` de cada aresta é calculado, em metros, pela soma das distâncias geodésicas entre os vértices da sua geometria.

Após a simplificação, **nós representam interseções e extremidades de vias, e arestas representam segmentos de rua entre elas**, o que corresponde à representação *primal* discutida no {ref}`cap-fundamentos-grafos`.

:::{note}
A simplificação altera substancialmente os números de nós e arestas, e portanto todas as métricas derivadas deles. Ao comparar redes ou reproduzir estudos, verifique sempre se o grafo foi simplificado (argumento `simplify=True`, o padrão) e com qual versão da OSMnx.
:::

## Preparação do ambiente

As dependências do livro estão listadas nos arquivos `requirements.txt` e `environment.yml` do repositório. Em um ambiente como o Google Colab, basta instalar a OSMnx e a mapclassify, que é usada pela GeoPandas para classificar valores em mapas temáticos:

In [ ]:
# Descomente a linha abaixo se estiver executando no Google Colab
# %pip install -q osmnx mapclassify folium

In [ ]:
import warnings

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox
import pandas as pd

# Mapa base para os mapas interativos: não exige chave de API e respeita
# as políticas de uso (os blocos do OpenStreetMap são bloqueados em ambientes como o Colab)
MAPA_BASE = "Esri.WorldGrayCanvas"

print("OSMnx", ox.__version__, "| NetworkX", nx.__version__, "| GeoPandas", gpd.__version__)

A OSMnx mantém um **cache** local das respostas das APIs. Com ele, a reexecução de um *notebook* não repete as consultas, o que torna a análise mais rápida, reduz a carga sobre servidores comunitários e aumenta a reprodutibilidade.

In [ ]:
ox.settings.use_cache = True      # reutiliza respostas já baixadas (pasta ./cache)
ox.settings.log_console = False   # mude para True para acompanhar as requisições

:::{tip}
As APIs Overpass e Nominatim são serviços públicos mantidos pela comunidade, com limites de uso. Evite consultas desnecessariamente grandes, mantenha o cache ativado e, para análises recorrentes, salve os grafos em disco (ver a seção {ref}`sec-persistencia`).
:::

## Delimitação da área de estudo

Toda extração começa pela definição de uma **área de estudo**. A OSMnx oferece três estratégias principais:

- **por nome de lugar**: geocodificação de um topônimo em um polígono (funções `*_from_place`);
- **por ponto e raio**: uma coordenada central e uma distância em metros (funções `*_from_point`);
- **por polígono ou retângulo envolvente**: geometria fornecida pelo usuário (funções `*_from_polygon` e `*_from_bbox`).

### Geocodificação de lugares

A **geocodificação** converte um nome ou endereço em coordenadas; neste caso, em um polígono de limite. A OSMnx utiliza o serviço Nominatim, do próprio OSM. O resultado depende da especificidade da consulta:

In [ ]:
municipio = ox.geocode_to_gdf("São Paulo, Brasil")
estado = ox.geocode_to_gdf("Estado de São Paulo, Brasil")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
estado.plot(ax=axes[0], color="lightblue", edgecolor="tab:blue")
municipio.plot(ax=axes[0], color="tab:red")
axes[0].set_title("Estado (azul) e município (vermelho)")
municipio.plot(ax=axes[1], color="lightblue", edgecolor="tab:blue")
axes[1].set_title("Município de São Paulo")
for ax in axes:
    ax.set_axis_off()
plt.show()

A consulta "São Paulo, Brasil" retornou o **município**. Para obter o estado, foi preciso explicitá-lo no texto. Ambiguidades desse tipo são comuns: nomes de bairros se repetem entre cidades, e o Nominatim pode retornar mais de um candidato. Inspecione sempre o polígono retornado antes de prosseguir.

A área de estudo deste e do próximo capítulo é o bairro da **Liberdade**, no centro de São Paulo, cuja malha viária densa e com muitas vias de mão única ilustra bem os conceitos em estudo.

In [ ]:
bairro = "Liberdade, São Paulo, Brasil"
area = ox.geocode_to_gdf(bairro)
area[["display_name", "geometry"]]

Para uma inspeção interativa, o método `explore` da GeoPandas gera um mapa Folium sobre um mapa base. Usamos o mapa base cinza da Esri, definido na constante `MAPA_BASE`, que não exige chave de API:

In [ ]:
area.explore(tiles=MAPA_BASE, style_kwds={"fillOpacity": 0.1, "weight": 2})

## Extração de feições geográficas

Além de redes viárias, a OSMnx extrai quaisquer feições do OSM por meio da família de funções `features_from_*`. O filtro é um dicionário de *tags* com a seguinte semântica:

- `{"building": True}` retorna todos os elementos que possuem a chave `building`, qualquer que seja o valor;
- `{"amenity": "hospital"}` retorna apenas os elementos com `amenity=hospital`;
- `{"amenity": ["hospital", "clinic"]}` retorna elementos com qualquer um dos valores da lista;
- um dicionário com várias chaves, como `{"leisure": "park", "landuse": "grass"}`, retorna a **união** dos elementos que satisfazem cada chave.

O resultado é um `GeoDataFrame` indexado por `(element, id)`, isto é, pelo tipo de elemento OSM (`node`, `way` ou `relation`) e pelo seu identificador, com uma coluna para cada *tag* encontrada.

In [ ]:
edificios = ox.features_from_place(bairro, tags={"building": True})
parques = ox.features_from_place(bairro, tags={"leisure": "park"})
pontos_onibus = ox.features_from_place(bairro, tags={"highway": "bus_stop"})
cursos_agua = ox.features_from_place(bairro, tags={"waterway": True})
equipamentos = ox.features_from_place(bairro, tags={"amenity": True})
hospitais = ox.features_from_place(bairro, tags={"amenity": "hospital"})

resumo = pd.DataFrame({
    "feições": [len(g) for g in (edificios, parques, pontos_onibus, cursos_agua, equipamentos, hospitais)],
    "atributos": [g.shape[1] for g in (edificios, parques, pontos_onibus, cursos_agua, equipamentos, hospitais)],
}, index=["edificações", "parques", "pontos de ônibus", "cursos d'água", "equipamentos (amenity)", "hospitais"])
resumo

:::{note}
Os números da tabela refletem o estado do OSM **no momento da execução** e mudam à medida que a comunidade edita o mapa. O número de colunas também varia, pois cada *tag* distinta presente nos dados gera uma coluna, em geral com muitos valores ausentes.
:::

Uma mesma consulta pode retornar geometrias de tipos diferentes. Hospitais, por exemplo, podem estar mapeados como pontos (um nó) ou como polígonos (o contorno do lote ou do prédio). Verificar os tipos de geometria evita surpresas em operações posteriores, como o cálculo de centroides ou de áreas:

In [ ]:
hospitais.geom_type.value_counts()

A {numref}`fig-feicoes-liberdade` sobrepõe as feições extraídas em um mapa estático.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
area.boundary.plot(ax=ax, color="black", linewidth=1)
edificios.plot(ax=ax, color="silver", edgecolor="gray", linewidth=0.2)
parques.plot(ax=ax, color="tab:green", alpha=0.6)
pontos_onibus.plot(ax=ax, color="tab:blue", markersize=12)
hospitais.representative_point().plot(ax=ax, color="tab:red", markersize=40, marker="P")

legenda = [
    plt.Line2D([0], [0], color="silver", lw=6, label="Edificações"),
    plt.Line2D([0], [0], color="tab:green", lw=6, alpha=0.6, label="Parques"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:blue", markersize=7, label="Pontos de ônibus"),
    plt.Line2D([0], [0], marker="P", color="w", markerfacecolor="tab:red", markersize=10, label="Hospitais"),
]
ax.legend(handles=legenda, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
ax.set_title("Feições do OpenStreetMap: Liberdade, São Paulo")
ax.set_axis_off()
plt.tight_layout()
plt.show()

Mapas interativos com várias camadas são construídos encadeando chamadas a `explore`, passando o mapa da chamada anterior no argumento `m`:

In [ ]:
mapa = area.explore(tiles=MAPA_BASE, name="Limite do bairro",
                    style_kwds={"fill": False, "color": "black", "weight": 2})
edificios[edificios.geom_type.isin(["Polygon", "MultiPolygon"])].explore(
    m=mapa, name="Edificações", color="gray", style_kwds={"weight": 0.3, "fillOpacity": 0.5})
parques.explore(m=mapa, name="Parques", color="green")
pontos_onibus.explore(m=mapa, name="Pontos de ônibus", color="blue", tooltip="name",
                      marker_kwds={"radius": 3})
folium.LayerControl().add_to(mapa)
mapa

:::{tip}
Adicionar feições uma a uma ao mapa com um laço sobre `iterrows()` é muito lento para milhares de edificações. Sempre que possível, passe o `GeoDataFrame` inteiro a `explore`, que gera uma única camada GeoJSON.
:::

## Obtenção da rede viária

### Tipos de rede

As funções `graph_from_*` recebem o argumento `network_type`, que seleciona um filtro pré-definido sobre as vias do OSM. A {numref}`tab-network-types` lista os tipos disponíveis na OSMnx 2.x.

:::{table} Valores do argumento network_type na OSMnx 2.x.
:name: tab-network-types

| `network_type` | Vias incluídas |
|---|---|
| `"drive"` | Vias públicas trafegáveis por automóveis; exclui vias de serviço (acessos a estacionamentos, vielas etc.) |
| `"drive_service"` | Como `"drive"`, mas inclui a maior parte das vias de serviço |
| `"walk"` | Vias públicas em que pedestres podem circular; ignora o sentido único (`oneway`), pois pedestres podem andar nos dois sentidos |
| `"bike"` | Vias públicas em que bicicletas podem circular |
| `"all_public"` | Todas as vias públicas em uso |
| `"all"` | Todas as vias em uso, públicas e privadas (padrão) |

:::

:::{warning}
Versões anteriores da OSMnx usavam nomes diferentes (por exemplo, `"all_private"`, equivalente ao atual `"all"`), e combinações como `"drive+bike"` **não existem**. Para filtros que não correspondem a nenhum tipo pré-definido, use o argumento `custom_filter`, que recebe uma expressão da linguagem Overpass QL, como mostramos adiante.
:::

Baixamos, a seguir, as redes para automóveis, pedestres e bicicletas:

In [ ]:
G_drive = ox.graph_from_place(bairro, network_type="drive")
G_walk = ox.graph_from_place(bairro, network_type="walk")
G_bike = ox.graph_from_place(bairro, network_type="bike")

comparacao = pd.DataFrame(
    {
        "nós (n)": [G.number_of_nodes() for G in (G_drive, G_walk, G_bike)],
        "arestas (m)": [G.number_of_edges() for G in (G_drive, G_walk, G_bike)],
        "extensão das arestas (km)": [
            sum(d["length"] for _, _, d in G.edges(data=True)) / 1000 for G in (G_drive, G_walk, G_bike)
        ],
    },
    index=["drive", "walk", "bike"],
).round(1)
comparacao

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ox.plot_graph(G_walk, ax=ax, node_size=0, edge_color="tab:blue", edge_linewidth=0.6,
              bgcolor="white", show=False, close=False)
ox.plot_graph(G_drive, ax=ax, node_size=0, edge_color="tab:red", edge_linewidth=0.9,
              bgcolor="white", show=False, close=False)
ax.legend(handles=[plt.Line2D([0], [0], color="tab:red", lw=2, label="drive"),
                   plt.Line2D([0], [0], color="tab:blue", lw=2, label="walk")],
          loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
ax.set_title("Redes viárias por modo: Liberdade")
plt.show()

A rede de pedestres é mais extensa porque inclui calçadas, escadarias, praças e passagens exclusivas, mas **não é um superconjunto exato** da rede de automóveis. Vias expressas, por exemplo, podem ser proibidas a pedestres. A escolha do `network_type` é, portanto, uma decisão de modelagem que deve corresponder ao fenômeno estudado.

### Rede a partir de um ponto e um raio

Quando a área de interesse não coincide com um limite administrativo, podemos definir um ponto central e um raio. O argumento `dist_type` controla a forma da área: `"bbox"` (padrão) usa um quadrado de meia-aresta `dist`, e `"network"` mantém apenas os nós a até `dist` metros do centro **pela rede**.

In [ ]:
centro_liberdade = (-23.5614, -46.6332)  # (latitude, longitude)
G_raio = ox.graph_from_point(centro_liberdade, dist=500, network_type="drive")

fig, ax = ox.plot_graph(G_raio, node_size=8, node_color="tab:red", edge_color="black",
                        edge_linewidth=0.5, bgcolor="white")

:::{note}
A OSMnx segue a convenção **(latitude, longitude)** em `graph_from_point`, mas armazena as coordenadas dos nós como `x` = longitude e `y` = latitude, a convenção cartesiana usada pelas bibliotecas de geometria. Trocar a ordem é um dos erros mais frequentes em análises espaciais: um ponto em São Paulo com coordenadas invertidas cai no oceano Índico.
:::

### Filtros personalizados

O argumento `custom_filter` permite selecionar vias por qualquer combinação de *tags*. O exemplo abaixo seleciona apenas as vias residenciais:

In [ ]:
G_residencial = ox.graph_from_place(bairro, custom_filter='["highway"="residential"]')
arestas_residenciais = ox.graph_to_gdfs(G_residencial, nodes=False)
arestas_residenciais.explore(tiles=MAPA_BASE, color="purple", tooltip="name",
                             style_kwds={"weight": 3})

### O efeito da simplificação topológica

Para observar a regra {eq}`eq-regra-simplificacao` em ação, comparamos o grafo obtido sem simplificação com o grafo simplificado:

In [ ]:
G_bruto = ox.graph_from_place(bairro, network_type="drive", simplify=False)

pd.DataFrame(
    {"nós": [len(G_bruto), len(G_drive)], "arestas": [G_bruto.number_of_edges(), G_drive.number_of_edges()]},
    index=["não simplificado", "simplificado"],
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, G, titulo in zip(axes, (G_bruto, G_drive), ("Não simplificado", "Simplificado")):
    ox.plot_graph(G, ax=ax, node_size=6, node_color="tab:red", edge_color="gray",
                  edge_linewidth=0.6, bgcolor="white", show=False, close=False)
    ax.set_title(f"{titulo}: n = {len(G)}")
plt.show()

A redução no número de nós é expressiva, mas a geometria das vias é preservada: a forma de cada segmento fundido fica armazenada no atributo `geometry` da aresta resultante. Métricas **topológicas**, como grau médio e densidade, são muito sensíveis a essa escolha; métricas **geométricas**, como a extensão total, não.

## Projeção cartográfica

Os grafos baixados usam coordenadas geográficas (latitude e longitude, em graus, no sistema WGS 84, código `EPSG:4326`). Graus não são unidades de comprimento: um grau de longitude mede cerca de 111 km no equador e zero nos polos. Assim, operações métricas como áreas, *buffers*, distâncias euclidianas e consultas de vizinho mais próximo exigem uma **projeção** para um sistema em metros.

A função `ox.project_graph` projeta o grafo para a zona UTM adequada à sua localização (ou para qualquer CRS informado em `to_crs`), atualizando os atributos `x`, `y` e `geometry`:

In [ ]:
G_drive_proj = ox.project_graph(G_drive)
print("CRS original: ", G_drive.graph["crs"])
print("CRS projetado:", G_drive_proj.graph["crs"])

no_exemplo = next(iter(G_drive.nodes))
print("Coordenadas (graus): ", G_drive.nodes[no_exemplo]["x"], G_drive.nodes[no_exemplo]["y"])
print("Coordenadas (metros):", G_drive_proj.nodes[no_exemplo]["x"], G_drive_proj.nodes[no_exemplo]["y"])

:::{important}
O atributo `length` das arestas é calculado pela OSMnx com distâncias geodésicas e está **sempre em metros**, mesmo no grafo não projetado. A projeção é necessária para operações geométricas sobre coordenadas, não para o uso de `length` como peso em algoritmos de caminho mínimo.
:::

## Conversões entre representações

### Grafo ↔ GeoDataFrames

A ponte entre a análise de grafos (NetworkX) e a análise espacial vetorial (GeoPandas) é feita por duas funções:

- `ox.graph_to_gdfs(G)` retorna um `GeoDataFrame` de nós (pontos, indexados pelo `osmid`) e um de arestas (linhas, indexadas por `(u, v, key)`);
- `ox.graph_from_gdfs(nos, arestas)` reconstrói o grafo a partir deles, o que permite editar atributos com a GeoPandas e voltar à NetworkX.

In [ ]:
nos, arestas = ox.graph_to_gdfs(G_drive)
print(nos.shape, arestas.shape)
arestas.head(3)

In [ ]:
G_reconstruido = ox.graph_from_gdfs(nos, arestas)
print("n:", len(G_drive), "→", len(G_reconstruido))
print("m:", G_drive.number_of_edges(), "→", G_reconstruido.number_of_edges())

### Multidígrafo → multigrafo e dígrafo

Algumas análises ignoram o sentido das vias (por exemplo, a orientação das ruas ou a acessibilidade de pedestres), e alguns algoritmos exigem grafos sem arestas paralelas. A OSMnx oferece conversões que tratam corretamente as particularidades das redes viárias:

- `ox.convert.to_undirected(G)` gera um `MultiGraph` e **funde** os pares de arcos opostos que representam a mesma via de mão dupla, evitando contá-la duas vezes;
- `ox.convert.to_digraph(G, weight="length")` gera um `DiGraph` e, entre arestas paralelas, mantém a de menor peso.

In [ ]:
G_u_osmnx = ox.convert.to_undirected(G_drive)
G_u_networkx = G_drive.to_undirected()

pd.DataFrame(
    {"arestas": [G_drive.number_of_edges(), G_u_osmnx.number_of_edges(), G_u_networkx.number_of_edges()]},
    index=["MultiDiGraph original", "ox.convert.to_undirected", "G.to_undirected (NetworkX)"],
)

:::{warning}
O método genérico `G.to_undirected()` da NetworkX, aplicado a um `MultiDiGraph`, preserva **ambos** os arcos de uma via de mão dupla como arestas paralelas. O resultado superestima a extensão da rede e distorce métricas baseadas em contagem de arestas. Para redes viárias, prefira `ox.convert.to_undirected`.
:::

(sec-persistencia)=
### Persistência: salvando e carregando grafos

Para evitar novas consultas às APIs e garantir que uma análise use exatamente os mesmos dados, salve o grafo em disco:

- **GraphML** (`ox.save_graphml` e `ox.load_graphml`) preserva integralmente a estrutura e os atributos e é o formato recomendado para reabrir o grafo em Python;
- **GeoPackage** (`ox.save_graph_geopackage`) grava camadas de nós e arestas legíveis em SIG como o QGIS.

In [ ]:
ox.save_graphml(G_drive, "dados/liberdade_drive.graphml")
ox.save_graph_geopackage(G_drive, "dados/liberdade_drive.gpkg")

G_carregado = ox.load_graphml("dados/liberdade_drive.graphml")
print(len(G_carregado), G_carregado.number_of_edges())

## Atributos dos nós

Os nós de um grafo da OSMnx carregam os atributos listados na {numref}`tab-atributos-nos`.

:::{table} Principais atributos dos nós em grafos da OSMnx.
:name: tab-atributos-nos

| Atributo | Descrição |
|---|---|
| `osmid` (índice) | Identificador do nó no OpenStreetMap |
| `x`, `y` | Longitude e latitude (ou coordenadas projetadas, após `project_graph`) |
| `street_count` | Número de ruas **físicas** que se encontram no nó, sem considerar o sentido |
| `highway` | *Tag* do nó OSM, quando existente: `traffic_signals`, `crossing`, `stop`, `mini_roundabout` etc. |
| `ref` | Referência do nó, por exemplo a numeração de uma saída de via expressa |

:::

### `street_count` não é o grau do nó

É essencial distinguir `street_count` do grau calculado pela NetworkX. No multidígrafo, uma rua de mão dupla que chega a um nó contribui com **dois** arcos, um de entrada e um de saída, de modo que $\deg(v) = \deg^-(v) + \deg^+(v)$ pode ser até o dobro do número de ruas. Já `street_count` conta cada rua física uma única vez, como faria um observador na esquina.

In [ ]:
graus = pd.DataFrame({
    "street_count": nx.get_node_attributes(G_drive, "street_count"),
    "grau_total": dict(G_drive.degree()),
    "grau_entrada": dict(G_drive.in_degree()),
    "grau_saida": dict(G_drive.out_degree()),
})
graus.head(8)

In [ ]:
pd.crosstab(graus["street_count"], graus["grau_total"])

A tabela cruzada mostra que um mesmo `street_count` corresponde a diferentes graus, conforme a combinação de ruas de mão única e de mão dupla em cada interseção. **Para caracterizar interseções, use `street_count`; para análises sobre o grafo orientado, use o grau.** Voltaremos a essa distinção no {ref}`cap-metricas-centralidade`.

### Exploração e visualização dos atributos

A distribuição de `street_count` resume a morfologia das interseções: nós com valor 1 são **becos sem saída**, nós com valor 3 são interseções em "T" e nós com valor 4 são cruzamentos em "+".

In [ ]:
nos["street_count"].value_counts().sort_index().rename("número de nós").to_frame()

In [ ]:
mapa = arestas.explore(tiles=MAPA_BASE, color="gray", style_kwds={"weight": 1, "opacity": 0.5})
nos.explore(m=mapa, column="street_count", categorical=True, cmap="Set1", legend=True,
            tooltip=["street_count", "highway"], marker_kwds={"radius": 4})

Os valores da coluna `highway` dos nós identificam elementos de controle de tráfego:

In [ ]:
nos["highway"].value_counts(dropna=True)

In [ ]:
semaforos = nos[nos["highway"] == "traffic_signals"]
mapa = arestas.explore(tiles=MAPA_BASE, color="gray", style_kwds={"weight": 1, "opacity": 0.5})
semaforos.explore(m=mapa, color="red", marker_kwds={"radius": 4}, tooltip="highway")

## Atributos das arestas

As arestas concentram a maior parte da informação semântica da rede. A {numref}`tab-atributos-arestas` descreve os principais atributos.

:::{table} Principais atributos das arestas em grafos da OSMnx.
:name: tab-atributos-arestas

| Atributo | Descrição |
|---|---|
| `u`, `v`, `key` (índice) | Nó de origem, nó de destino e chave que distingue arestas paralelas |
| `osmid` | Identificador(es) do(s) *way(s)* OSM que compõem o segmento |
| `highway` | Classe da via: `primary`, `secondary`, `tertiary`, `residential`, sufixo `_link` para alças de acesso etc. |
| `oneway` | `True` se a via é de mão única |
| `reversed` | `True` se o arco foi criado no sentido oposto ao de digitalização do *way* no OSM |
| `length` | Comprimento do segmento, em metros |
| `name`, `ref` | Nome e código da via |
| `maxspeed`, `lanes`, `width` | Velocidade máxima, número de faixas e largura (quando mapeados) |
| `bridge`, `tunnel`, `junction`, `access` | Ponte ou viaduto, túnel, tipo de junção (por exemplo, rotatória) e restrições de acesso |
| `geometry` | Geometria `LineString` completa do segmento |

:::

In [ ]:
print("Extensão total das arestas (km):", round(arestas["length"].sum() / 1000, 2))
print("\nPresença de cada atributo (% de arestas com valor):")
(arestas.notna().mean() * 100).round(1).sort_values(ascending=False)

### Atributos com múltiplos valores

Ao fundir segmentos durante a simplificação, a OSMnx pode reunir *ways* com valores diferentes para uma mesma *tag*. Quando isso ocorre, o atributo da aresta resultante passa a ser uma **lista**, por exemplo `maxspeed = ['30', '50']`. Além disso, *tags* numéricas do OSM são armazenadas como **texto**. Antes de qualquer análise quantitativa, esses atributos precisam ser normalizados.

In [ ]:
def contar_tipos(serie):
    """Conta quantos valores de uma série são texto, lista, ausentes ou de outro tipo."""
    def tipo(valor):
        if isinstance(valor, list):
            return "lista"
        if isinstance(valor, str):
            return "texto"
        if pd.isna(valor):
            return "ausente"
        return type(valor).__name__
    return serie.map(tipo).value_counts()


pd.DataFrame({col: contar_tipos(arestas[col]) for col in ["highway", "maxspeed", "lanes", "name"]
              if col in arestas}).fillna(0).astype(int)

Uma estratégia comum é converter cada valor em número e, no caso de listas, adotar uma regra explícita de agregação, como o máximo, a média ou o primeiro valor:

In [ ]:
def para_numero(valor, agregacao=max):
    """Converte '50', ['30', '50'] ou '50 mph' em número; retorna NaN se não for possível."""
    valores = valor if isinstance(valor, list) else [valor]
    numeros = []
    for v in valores:
        try:
            numeros.append(float(str(v).split()[0]))
        except (TypeError, ValueError, IndexError):
            continue
    return agregacao(numeros) if numeros else float("nan")


if "maxspeed" in arestas:
    arestas["maxspeed_kmh"] = arestas["maxspeed"].map(para_numero)
    print(arestas["maxspeed_kmh"].describe())

Com o atributo normalizado, filtros e mapas temáticos tornam-se diretos:

In [ ]:
if "maxspeed_kmh" in arestas:
    display(arestas.dropna(subset=["maxspeed_kmh"]).explore(tiles=MAPA_BASE,
        column="maxspeed_kmh", cmap="plasma", legend=True,
        tooltip=["name", "maxspeed", "maxspeed_kmh"], style_kwds={"weight": 3}))

:::{note}
Se a unidade não for informada, o OSM assume km/h para `maxspeed`. Valores como `"50 mph"` ou `"BR:urban"` ocorrem em algumas regiões; a função acima lida com o primeiro caso, mas ignora o segundo (retorna `NaN`). No {ref}`cap-roteamento`, a função `ox.add_edge_speeds` imputará velocidades ausentes a partir da classe da via.
:::

O mesmo raciocínio vale para atributos categóricos. Para colorir as arestas pela classe da via, convertemos as listas em texto:

In [ ]:
arestas["highway_str"] = arestas["highway"].map(lambda v: " | ".join(v) if isinstance(v, list) else v)
arestas.explore(tiles=MAPA_BASE, column="highway_str", cmap="tab10", legend=True,
                tooltip=["name", "highway_str", "length"], style_kwds={"weight": 3})

### Seleção de arestas por atributo

Existem dois caminhos equivalentes para selecionar subconjuntos da rede: filtrar o `GeoDataFrame` de arestas ou construir um **subgrafo** com `edge_subgraph`. O primeiro é adequado para mapas e tabelas; o segundo preserva a estrutura de grafo e permite aplicar algoritmos ao subconjunto.

In [ ]:
def eh_ponte(valor):
    valores = valor if isinstance(valor, list) else [valor]
    return any(v in ("yes", "viaduct") for v in valores)


# Opção 1: filtro no GeoDataFrame
if "bridge" in arestas:
    pontes = arestas[arestas["bridge"].map(eh_ponte)]
    print("Arestas em pontes/viadutos:", len(pontes))

# Opção 2: subgrafo
arestas_ponte = [(u, v, k) for u, v, k, d in G_drive.edges(keys=True, data=True) if eh_ponte(d.get("bridge"))]
G_pontes = G_drive.edge_subgraph(arestas_ponte)
print("Subgrafo de pontes: n =", len(G_pontes), "| m =", G_pontes.number_of_edges())

In [ ]:
if len(arestas_ponte) > 0:
    mapa = arestas.explore(tiles=MAPA_BASE, color="gray", style_kwds={"weight": 1, "opacity": 0.4})
    display(ox.graph_to_gdfs(G_pontes, nodes=False).explore(m=mapa, color="red", tooltip=["name", "bridge"],
                                                    style_kwds={"weight": 4}))

## Síntese do capítulo

- O OSM modela o mundo com **nós, *ways* e relações**, descritos por **tags** `chave=valor`. Seus dados são abertos, mas heterogêneos em completude.
- A OSMnx converte os *ways* com a chave `highway` em um **multidígrafo** primal: arcos em ambos os sentidos para vias de mão dupla, chaves para arestas paralelas e **simplificação topológica** que mantém apenas interseções e extremidades.
- A área de estudo pode ser definida por nome (geocodificação), por ponto e raio ou por polígono. O `network_type` e o `custom_filter` definem quais vias compõem a rede.
- Operações geométricas exigem **projeção** para um CRS métrico; o atributo `length` está sempre em metros.
- `street_count` **não** é o grau do nó; `ox.convert.to_undirected` é preferível a `G.to_undirected()` para redes viárias.
- Atributos do OSM podem ser **listas** e **textos**, e devem ser normalizados antes de análises quantitativas.

## Exercícios Práticos e Desafios

### Exercícios conceituais

1. Uma rua de mão dupla com 5 nós OSM (dos quais apenas os dois extremos são interseções) gera quantos arcos no grafo **não simplificado**? E no **simplificado**? E se a rua for de mão única?
2. Explique por que a razão $m/n$ de um grafo viário da OSMnx costuma ser maior que a razão correspondente calculada sobre o mesmo grafo convertido com `ox.convert.to_undirected`.
3. Por que o `network_type="walk"` ignora a *tag* `oneway`? Dê um exemplo de situação em que essa escolha poderia levar a um resultado inadequado.

### Exercícios computacionais

4. Escolha um bairro da sua cidade e produza uma tabela análoga à {numref}`tab-network-types`, com $n$, $m$ e extensão total para cada `network_type`. Discuta as diferenças.
5. Para o bairro escolhido, extraia escolas (`amenity=school`) e calcule quantas estão mapeadas como ponto e quantas como polígono. Converta todas para pontos com `representative_point()` e visualize-as sobre a rede `walk`.
6. Construa um `custom_filter` que selecione apenas as vias arteriais (`primary`, `secondary` e `tertiary`, incluindo suas alças `_link`). Compare a extensão dessa rede com a da rede `drive` completa.
7. Salve a rede do bairro em GeoPackage, abra o arquivo no QGIS e verifique se os atributos `length`, `oneway` e `highway` foram preservados.

### Desafios

8. **Qualidade dos dados.** Para três bairros com perfis socioeconômicos distintos, calcule a porcentagem de arestas com `maxspeed`, `lanes` e `name` preenchidos. Discuta a hipótese de que a completude do OSM está associada ao perfil do bairro {cite:p}`haklay2008`.
9. **Consolidação de interseções.** Grandes cruzamentos com canteiros centrais aparecem como vários nós próximos. Use `ox.consolidate_intersections` sobre o grafo projetado, com tolerâncias de 5, 10 e 15 metros, e analise como $n$ e a distribuição de `street_count` variam. Qual tolerância parece mais adequada para a Liberdade?